# Document Structure for RAG

This notebook explains the `Document` object used in LangChain and shows metadata practices that make retrieval better and easier to debug.

Goals:
- Understand `page_content` and `metadata`
- Build clean, consistent metadata
- Validate document quality before indexing

In [ ]:
from datetime import datetime, UTC
from langchain_core.documents import Document


def validate_document(doc: Document) -> list[str]:
    issues = []
    if not isinstance(doc.page_content, str) or not doc.page_content.strip():
        issues.append("page_content must be a non-empty string")

    required_metadata = ["source", "doc_type", "created_at"]
    for key in required_metadata:
        if key not in doc.metadata:
            issues.append(f"Missing metadata key: {key}")

    if "source" in doc.metadata and not str(doc.metadata["source"]).strip():
        issues.append("metadata['source'] must not be empty")

    return issues

In [ ]:
good_doc = Document(
    page_content="RAG combines retrieval and generation to produce grounded answers.",
    metadata={
        "source": "sample_dataset.txt",
        "doc_type": "text",
        "created_at": datetime.now(UTC).isoformat(),
        "chunk_id": "text-0001",
        "topic": "rag-intro",
    },
)

bad_doc = Document(
    page_content="   ",
    metadata={"source": "", "doc_type": "text"},
)

for name, doc in [("good_doc", good_doc), ("bad_doc", bad_doc)]:
    problems = validate_document(doc)
    print(f"\n{name}")
    print("metadata:", doc.metadata)
    print("validation:", "PASS" if not problems else "FAIL")
    for p in problems:
        print("-", p)

## Industry Notes

Use metadata as a contract. At minimum keep:
- source
- doc_type
- created_at
- chunk_id (after chunking)

Why this matters:
- Better filtering in retrieval
- Easier debugging and traceability
- Safer re-indexing workflows